<a href="https://colab.research.google.com/github/ext-colorful/LangChain-Essentials/blob/main/%F0%9F%AA%9BCustomize_Your_Agent%F0%9F%A4%96L8_dynamic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[课程地址](https://academy.langchain.com/courses/take/langchain-essentials-python/lessons/69388340-lesson-8-dynamic-prompt)😁
[源码地址](https://github.com/langchain-ai/lca-langchainV1-essentials/blob/main/python/L1_fast_agent.ipynb)😁
[LANGSMITH官网](https://smith.langchain.com)😁
[LANGCHAIN智能助手](https://chat.langchain.com)😁
[免费的智能体代理商](https://api.chatanywhere.tech)

Lessons 2–7 covered out-of-the-box features. However, create_agent also supports both prebuilt and user-defined customization through Middleware. This section describes middleware and includes two lessons highlighting specific use cases.
> 课程2-7节涵盖了开箱即用的功能。然而，create_agent也通过中间件支持预构建和用户定义的自定义。本节描述了中间件，并包括两个突出具体用例的课程。

## Learn how to dynamically modify the agent’s system prompt to react to changing contexts.
> 学习如何动态修改代理的系统提示以应对不断变化的环境。

# Dynamic Prompt
![链接文字](https://github.com/langchain-ai/lca-langchainV1-essentials/blob/main/python/assets/LC_DynamicPrompts.png?raw=true)

## Setup
Load and/or check for needed environmental variables
> 加载和/或检查所需的环境变量

In [1]:
!pip install -q langgraph==1.0.3 langchain==1.0.8 langchain-openai==1.0.3 langchain-community==0.4.1 langgraph-cli[inmem]==0.4.7 langchain-mcp-adapters==0.1.13
# Used to securely store your API key
# 用于安全存储您的API密钥
from google.colab import userdata
import os

# Retrieve the API key from Colab secrets
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_BASE"] = userdata.get('OPENAI_API_BASE')
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain_agent_L8_dynamic"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.7/93.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 473.8/473.8 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

In [2]:
!git clone https://github.com/langchain-ai/lca-langchainV1-essentials.git
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///lca-langchainV1-essentials/python/Chinook.db")

Cloning into 'lca-langchainV1-essentials'...
remote: Enumerating objects: 563, done.
remote: Counting objects: 100% (184/184), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 563 (delta 166), reused 128 (delta 128), pack-reused 379 (from 1)
Receiving objects: 100% (563/563), 3.33 MiB | 25.27 MiB/s, done.
Resolving deltas: 100% (386/386), done.


In [3]:
from dataclasses import dataclass

@dataclass
class RuntimeContext:
    is_employee: bool
    db: SQLDatabase

In [4]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [5]:
SYSTEM_PROMPT_TEMPLATE = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
{table_limits}
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

## Build a Dynamic Prompt
Utilize runtime context and middleware to generate a dynamic prompt.
> 利用运行时上下文和中间件来生成动态提示。

In [6]:
from langchain.agents.middleware.types import ModelRequest, dynamic_prompt

@dynamic_prompt
def dynamic_system_prompt(request: ModelRequest) -> str:
    if not request.runtime.context.is_employee:
        table_limits = "- Limit access to these tables: Album, Artist, Genre, Playlist, PlaylistTrack, Track."
        # 限制访问这些表：专辑、艺术家、流派、播放列表、播放列表曲目、曲目。
    else:
        table_limits = ""

    return SYSTEM_PROMPT_TEMPLATE.format(table_limits=table_limits)

Include middleware in `create_agent`.
> 在 `create_agent` 中包含中间件。


In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-3.5-turbo",
    tools=[execute_sql],
    middleware=[dynamic_system_prompt],
    context_schema=RuntimeContext,
)

In [8]:
question = "What is the most costly purchase by Frank Harris?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(is_employee=False, db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

/usr/local/lib/python3.12/dist-packages/pydantic/v1/main.py:1054: UserWarning: LangSmith now uses UUID v7 for run and trace identifiers. This warning appears when passing custom IDs. Please use: from langsmith import uuid7
            id = uuid7()
Future versions will require UUID v7.
  input_data = validator(cls_, input_data)


================================ Human Message =================================

What is the most costly purchase by Frank Harris?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_Uk1lfFGNfiZI8nvyM29MYtQI)
 Call ID: call_Uk1lfFGNfiZI8nvyM29MYtQI
  Args:
    query: SELECT CustomerId, FirstName, LastName, Total FROM Invoice JOIN Customer ON Invoice.CustomerId = Customer.CustomerId WHERE FirstName = 'Frank' AND LastName = 'Harris' ORDER BY Total DESC LIMIT 1;
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) ambiguous column name: CustomerId
[SQL: SELECT CustomerId, FirstName, LastName, Total FROM Invoice JOIN Customer ON Invoice.CustomerId = Customer.CustomerId WHERE FirstName = 'Frank' AND LastName = 'Harris' ORDER BY Total DESC LIMIT 1;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
================================== Ai Mess

In [9]:
question = "What is the most costly purchase by Frank Harris?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(is_employee=True, db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the most costly purchase by Frank Harris?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_LAUYmSKFPYLmk3tKGQbWEtYJ)
 Call ID: call_LAUYmSKFPYLmk3tKGQbWEtYJ
  Args:
    query: SELECT purchaser_name, item_name, price FROM purchases WHERE purchaser_name = 'Frank Harris' ORDER BY price DESC LIMIT 1;
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) no such table: purchases
[SQL: SELECT purchaser_name, item_name, price FROM purchases WHERE purchaser_name = 'Frank Harris' ORDER BY price DESC LIMIT 1;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_mLtTEG9eTk7dgXWv9TDfRpqE)
 Call ID: call_mLtTEG9eTk7dgXWv9TDfRpqE
  Args:
    query: S

In [10]:
question = "Frank Harris最昂贵的购买是什么？"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(is_employee=True, db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Frank Harris最昂贵的购买是什么？
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_5ksfVTLXX9r67bjbwAhxiswf)
 Call ID: call_5ksfVTLXX9r67bjbwAhxiswf
  Args:
    query: SELECT Purchases.purchase_name, Purchases.price FROM Purchases JOIN Customers ON Purchases.customer_id = Customers.customer_id WHERE Customers.customer_name = 'Frank Harris' ORDER BY Purchases.price DESC LIMIT 1;
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) no such table: Purchases
[SQL: SELECT Purchases.purchase_name, Purchases.price FROM Purchases JOIN Customers ON Purchases.customer_id = Customers.customer_id WHERE Customers.customer_name = 'Frank Harris' ORDER BY Purchases.price DESC LIMIT 1;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
================================== Ai Messag